# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [2]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [3]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def morganfp_1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

def rdkitfp(mol):
    fp = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

fpgens = {
    "MorganFP": morganfp,
    "MorganFP_1024": morganfp_1024,
    "MACCSkeys": maccskeys,
    "RDKitFP": rdkitfp,
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MorganFP_1024,MACCSkeys,RDKitFP
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x11f02e810>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x11f02e8f0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x11f02e960>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x11f02e9d0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x11f02ea40>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [4]:
X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

In [5]:
# Split the data into training and testing sets, ensuring that the class distribution is preserved (stratified sampling).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Convert the data to PyTorch tensors for use in the neural network. There are many ways to do this, but here we convert the entire dataset at once. 
# check the lecture session for other approach, here we use this and give a datatyepe to ensure that the data is in the correct format for the model.
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)


4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [37]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(2048, 512) #2048 input nodes, 512 output nodes fc1: fully connected layer1
        self.fc2 = nn.Linear(512, 128) #512 input nodes, 128 output nodes
        self.fc3 = nn.Linear(128, 32) #128 input nodes, 32 output nodes
        self.output = nn.Linear(32, 1)#32 input nodes, 1 output node (binary classification)
        #this model workflow  is: 2048 -> 512 -> 128 -> 32 -> 1, which is a common architecture for a feedforward neural network. 
        # The number of nodes in each layer decreases as we go deeper into the network,
        #  which can help the model learn more abstract features from the data.
        self.dropout = nn.Dropout(0.5) # Dropout layer with a dropout rate of 0.3 to prevent overfitting.

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x)) # Apply ReLU activation function after the first fully connected layer.
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = torch.sigmoid(self.output(x))

        return x


### Hyperparameters tunning

In [38]:
# Parameters (change and add as needed)
learning_rate = 0.001
num_epochs = 50

***predict → criterion says "you were this wrong" → optimizer nudges weights → repeat***

In [39]:
model = BinClassifierNN()

# choose a loss function for the classification
criterion =nn.BCELoss() # Binary Cross Entropy Loss, since this is a binary classification problem.
# it measures the difference between the predicted probabilities and the actual binary labels (0 or 1).

# choose an optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [40]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.5862
Epoch [20/100], Loss: 0.4272
Epoch [30/100], Loss: 0.2875
Epoch [40/100], Loss: 0.1909
Epoch [50/100], Loss: 0.1179


6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [41]:
from sklearn.metrics import accuracy_score, roc_auc_score
# Evaluate the model on the test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test).squeeze()
    test_loss = criterion(test_outputs, y_test)
    
    y_pred_probs = test_outputs
    y_pred = (y_pred_probs >= 0.5).float()

y_true_np = np.array(y_test.tolist())
y_pred_np = np.array(y_pred.tolist())
y_probs_np = np.array(y_pred_probs.tolist())

print(f"Test Loss (BCE): {test_loss.item():.4f}")
print(f"Test Accuracy:   {accuracy_score(y_true_np, y_pred_np):.4f}")
print(f"Test ROC-AUC:    {roc_auc_score(y_true_np, y_probs_np):.4f}")
#old resukts of model in assignment 3
print("\n--- Baseline comparison ---")
print(f"KNN:               Accuracy 0.79 | ROC-AUC 0.86")
print(f"Decision Tree:     Accuracy 0.78 | ROC-AUC 0.77")
print(f"Random Forest:     Accuracy 0.83 | ROC-AUC 0.90")
print(f"Gradient Boosting: Accuracy 0.77 | ROC-AUC 0.85")

Test Loss (BCE): 0.7605
Test Accuracy:   0.7912
Test ROC-AUC:    0.8598

--- Baseline comparison ---
KNN:               Accuracy 0.79 | ROC-AUC 0.86
Decision Tree:     Accuracy 0.78 | ROC-AUC 0.77
Random Forest:     Accuracy 0.83 | ROC-AUC 0.90
Gradient Boosting: Accuracy 0.77 | ROC-AUC 0.85


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [42]:
# Save only the learned weights (recommended)
torch.save(model.state_dict(), "mutagenicity_nn_weights.pth")

print("Model weights saved successfully.")

Model weights saved successfully.


### MACCS Keys

In [12]:
# --- Train on MACCS keys (167 bits) ---

# Step 3: Prepare data
X_maccs = np.stack(df["MACCSkeys"].values)
y = df["mutagenicity"].to_numpy()

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_maccs, y, test_size=0.2, random_state=42, stratify=y
)

X_train_m = torch.tensor(X_train_m, dtype=torch.float32)
X_test_m = torch.tensor(X_test_m, dtype=torch.float32)
y_train_m = torch.tensor(y_train_m, dtype=torch.float32)
y_test_m = torch.tensor(y_test_m, dtype=torch.float32)

In [18]:
# Build and train (smaller network for 167 bits)
class BinClassifierNN_MACCS(nn.Module):
    def __init__(self):
        super(BinClassifierNN_MACCS, self).__init__()
        self.fc1 = nn.Linear(167, 64)
        self.fc2 = nn.Linear(64, 16)
        self.output = nn.Linear(16, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.output(x))
        return x

model_maccs = BinClassifierNN_MACCS()
criterion_maccs = nn.BCELoss()
optimizer_maccs = optim.Adam(model_maccs.parameters(), lr=0.001)

for epoch in range(100):
    model_maccs.train()
    outputs = model_maccs(X_train_m).squeeze()
    loss = criterion_maccs(outputs, y_train_m)
    optimizer_maccs.zero_grad()
    loss.backward()
    optimizer_maccs.step()
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

Epoch [20/100], Loss: 0.6543
Epoch [40/100], Loss: 0.6046
Epoch [60/100], Loss: 0.5590
Epoch [80/100], Loss: 0.5207
Epoch [100/100], Loss: 0.4885


### Hyperparameters tunning

In [31]:
# Parameters (change and add as needed)
learning_rate = 0.0001
num_epochs_maccs = 500

In [32]:
model_maccs = BinClassifierNN_MACCS()
criterion_maccs = nn.BCELoss() #binary cross entropy loss, since this is a binary classification problem.
#it measures the difference between the predicted probabilities and the actual binary labels (0 or 1).

#choose an optimizer
optimizer_maccs = optim.Adam(model_maccs.parameters(), lr=learning_rate)  

In [33]:
for epoch in range(num_epochs_maccs):
    model_maccs.train()
    outputs = model_maccs(X_train_m).squeeze()
    loss = criterion_maccs(outputs, y_train_m)
    optimizer_maccs.zero_grad()
    loss.backward()
    optimizer_maccs.step()
    if (epoch + 1) % 30 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs_maccs}], Loss: {loss.item():.4f}')

Epoch [30/500], Loss: 0.6929
Epoch [60/500], Loss: 0.6833
Epoch [90/500], Loss: 0.6709
Epoch [120/500], Loss: 0.6590
Epoch [150/500], Loss: 0.6426
Epoch [180/500], Loss: 0.6286
Epoch [210/500], Loss: 0.6162
Epoch [240/500], Loss: 0.6038
Epoch [270/500], Loss: 0.5943
Epoch [300/500], Loss: 0.5843
Epoch [330/500], Loss: 0.5755
Epoch [360/500], Loss: 0.5691
Epoch [390/500], Loss: 0.5630
Epoch [420/500], Loss: 0.5566
Epoch [450/500], Loss: 0.5446
Epoch [480/500], Loss: 0.5416


In [36]:
# Step 6: Evaluate MACCS model
model_maccs.eval()
with torch.no_grad():
    test_out = model_maccs(X_test_m).squeeze()
    test_loss_maccs = criterion_maccs(test_out, y_test_m)  
    y_probs = np.array(test_out.tolist())
    y_preds = (y_probs >= 0.5).astype(int)
    y_true = np.array(y_test_m.tolist())


print(f"MACCS - Train Loss: {loss.item():.4f}")
print(f"MACCS - Test Loss:  {test_loss_maccs.item():.4f}")
print(f"MACCS - Accuracy:   {accuracy_score(y_true, y_preds):.4f}")
print(f"MACCS - ROC-AUC:    {roc_auc_score(y_true, y_probs):.4f}")

MACCS - Train Loss: 0.5367
MACCS - Test Loss:  0.5273
MACCS - Accuracy:   0.7706
MACCS - ROC-AUC:    0.8221


#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
- The NN achieved Accuracy 0.78 and ROC-AUC 0.84 on Morgan FPs. This is comparable to KNN (0.79/0.86) and Gradient Boosting (0.77/0.85), but clearly below Random Forest (0.83/0.90). The NN did not outperform the simpler models here, which is expected for small tabular datasets with a few thousand samples, ensemble methods like Random Forest often perform just as well or better than neural networks, with far less tuning effort. NNs typically shine on much larger datasets or with more complex input types (images, sequences).
2) Did you observe any difference between different fingerprint types?
- Yes, the MORGAN FP performed better than the MACCS keys (check the Accuracy and ROC-AUC values), This is because Morgan fingerprints capture detailed local chemical environments, while MACCS keys are based on a limited set of predefined structural patterns. Therefore, Morgan fingerprints provide richer information for the model (especially that it Morgan FP contain valuable information which relate to toxicity = mutagenicity such as aromatic amines and nitrogroups).
3) Did the fingerprint size impact the model prediction? What message is to be learned from this
- There is no big impact when increasing the fingerprint size, it provides more detail but also increases sparsity and model complexity. Beyond a certain point, this leads to bad returns and may increase overfitting.
- Bigger is not always better!
4) What were some model parameters for decent performance depending on the fingerprint type? 
- A funnel-shaped architecture worked well (e.g., 2048→512→128→32→1) with ReLU activations, dropout (0.5), and the Adam optimizer with a learning rate of 0.001. Smaller fingerprints like MACCS required smaller networks (167→64→16→1). Matching model size to input size was important for good performance.
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
- Yes, significantly. Training loss dropped to 0.008 while test loss was 1.80, a clear sign that the model memorized the training data instead of learning generalizable chemical patterns. I applied Dropout 0.5 and number of epochs 50 to reduce overfitting. Other approaches that could help: early stopping (stop training when test loss stops improving).

6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
- Mutagenicity is typically measured using the Ames test. Noise in this context includes experimental variability, differences in testing conditions, and borderline compounds. These factors make prediction difficult. In practice, such models can be used for virtual screening to identify potentially mutagenic compounds early. Other useful QSAR tools include graph neural networks and uncertainty estimation methods.
7) Why is exporting a full model usually not recommended?
- Saving the full model is not recommended because it depends on the exact code and environment, making it less portable and potentially unsafe. Instead, saving only the model parameters (state_dict) is preferred, as it is more flexible, reproducible, and secure.

### In class solution discussion

In [1]:
#dropout helps prevent overfitting because it randomly "drops out" a fraction of the neurons during training, which forces the model to learn more robust features and prevents 
# it from relying too heavily on any single neuron. This can lead to better generalization on unseen data.